In [2]:
# ============================================================
# DATASET: German Credit
# Purpose: Hyperparameter tuning (grid search) + full 5-fold CV
# evaluation, closing the two methodology gaps against the
# approved proposal (Section 7.3, 7.5)
# ============================================================

import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier

SCALED_DIR = Path("../Data/Processed")
RANDOM_SEED = 42

def clean_column_names(df):
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

X_german_train = pd.read_csv(SCALED_DIR / "german_X_train.csv")
clean_column_names(X_german_train)
X_german_test = pd.read_csv(SCALED_DIR / "german_X_test.csv")
clean_column_names(X_german_test)
y_german_train = pd.read_csv(SCALED_DIR / "german_y_train.csv").squeeze()
y_german_test = pd.read_csv(SCALED_DIR / "german_y_test.csv").squeeze()

# Recreate the same 5-fold split used throughout the project (deterministic,
# same seed and shuffle setting as the folds saved back in Step 2)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(X_german_train.shape, X_german_test.shape)

(800, 40) (200, 40)


In [3]:
# Custom scorers for G-mean and MCC (not built into sklearn by default)
scoring = {
    "AUC-ROC": "roc_auc",
    "F1": "f1",
    "G-mean": make_scorer(geometric_mean_score),
    "MCC": make_scorer(matthews_corrcoef)
}

# Parameter grids exactly as specified in the approved proposal (Section 7.5)
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2"]
}

xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 6, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

print("Scoring metrics and parameter grids defined")

Scoring metrics and parameter grids defined


In [4]:
# ============================================================
# DATASET: German Credit
# Step: Grid search RF, no resampling, optimizing for F1 (minority class)
# ============================================================

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED),
    param_grid=rf_param_grid,
    scoring="f1",          # optimizing for minority-class F1, not AUC
    cv=skf,
    n_jobs=-1              # use all available CPU cores to speed this up
)
rf_grid.fit(X_german_train, y_german_train)

print("Best RF params (no resampling):", rf_grid.best_params_)
print("Best CV F1 score:", rf_grid.best_score_)

Best RF params (no resampling): {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}
Best CV F1 score: 0.48142888412709384


In [5]:
# ============================================================
# DATASET: German Credit
# Step: Grid search RF + SMOTE
# Note: param names need the "clf__" prefix since RF is now
# inside a pipeline alongside the SMOTE step
# ============================================================

rf_smote_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED))
])

rf_smote_param_grid = {f"clf__{k}": v for k, v in rf_param_grid.items()}

rf_smote_grid = GridSearchCV(
    rf_smote_pipe,
    param_grid=rf_smote_param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1
)
rf_smote_grid.fit(X_german_train, y_german_train)

print("Best RF+SMOTE params:", rf_smote_grid.best_params_)
print("Best CV F1 score:", rf_smote_grid.best_score_)

Best RF+SMOTE params: {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__n_estimators': 100}
Best CV F1 score: 0.5911051809528756


In [6]:
# ============================================================
# DATASET: German Credit
# Step: Grid search RF + SMOTE-ENN
# ============================================================

rf_smoteenn_pipe = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED))
])

rf_smoteenn_grid = GridSearchCV(
    rf_smoteenn_pipe,
    param_grid=rf_smote_param_grid,  # same prefixed grid, reused
    scoring="f1",
    cv=skf,
    n_jobs=-1
)
rf_smoteenn_grid.fit(X_german_train, y_german_train)

print("Best RF+SMOTE-ENN params:", rf_smoteenn_grid.best_params_)
print("Best CV F1 score:", rf_smoteenn_grid.best_score_)

Best RF+SMOTE-ENN params: {'clf__max_depth': None, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 5, 'clf__n_estimators': 200}
Best CV F1 score: 0.6017375583727442


In [7]:
# ============================================================
# DATASET: German Credit
# Step: Grid search XGBoost, no resampling
# ============================================================

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"),
    param_grid=xgb_param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1
)
xgb_grid.fit(X_german_train, y_german_train)

print("Best XGBoost params (no resampling):", xgb_grid.best_params_)
print("Best CV F1 score:", xgb_grid.best_score_)

Best XGBoost params (no resampling): {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1.0}
Best CV F1 score: 0.5787163561076605


In [8]:
# ============================================================
# DATASET: German Credit
# Step: Grid search XGBoost + SMOTE
# ============================================================

xgb_smote_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])

xgb_smote_param_grid = {f"clf__{k}": v for k, v in xgb_param_grid.items()}

xgb_smote_grid = GridSearchCV(
    xgb_smote_pipe,
    param_grid=xgb_smote_param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1
)
xgb_smote_grid.fit(X_german_train, y_german_train)

print("Best XGBoost+SMOTE params:", xgb_smote_grid.best_params_)
print("Best CV F1 score:", xgb_smote_grid.best_score_)

Best XGBoost+SMOTE params: {'clf__colsample_bytree': 1.0, 'clf__learning_rate': 0.01, 'clf__max_depth': 6, 'clf__n_estimators': 100, 'clf__subsample': 1.0}
Best CV F1 score: 0.6070145310871558


In [9]:
# ============================================================
# DATASET: German Credit
# Step: Grid search XGBoost + SMOTE-ENN
# ============================================================

xgb_smoteenn_pipe = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])

xgb_smoteenn_grid = GridSearchCV(
    xgb_smoteenn_pipe,
    param_grid=xgb_smote_param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1
)
xgb_smoteenn_grid.fit(X_german_train, y_german_train)

print("Best XGBoost+SMOTE-ENN params:", xgb_smoteenn_grid.best_params_)
print("Best CV F1 score:", xgb_smoteenn_grid.best_score_)

Best XGBoost+SMOTE-ENN params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 300, 'clf__subsample': 0.8}
Best CV F1 score: 0.6103666505792327


In [10]:
# ============================================================
# DATASET: German Credit
# Step: Stacked ensemble using tuned base models, full CV evaluation
# for all six configurations (RF/XGB/Stack x no-resample/SMOTE/SMOTE-ENN)
# ============================================================

def cv_evaluate(model, X, y, cv, name):
    # Runs full cross-validation and returns mean +/- std for each metric
    results = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return {
        "config": name,
        "AUC-ROC": f"{results['test_AUC-ROC'].mean():.3f} +/- {results['test_AUC-ROC'].std():.3f}",
        "F1": f"{results['test_F1'].mean():.3f} +/- {results['test_F1'].std():.3f}",
        "G-mean": f"{results['test_G-mean'].mean():.3f} +/- {results['test_G-mean'].std():.3f}",
        "MCC": f"{results['test_MCC'].mean():.3f} +/- {results['test_MCC'].std():.3f}"
    }

# Extract best params (without the "clf__" prefix) for use in the stack
rf_best_params = rf_grid.best_params_
xgb_best_params = xgb_grid.best_params_

# Tuned RF and XGBoost (no resampling) - re-evaluated via full CV for the final table
result_rf_final = cv_evaluate(
    RandomForestClassifier(random_state=RANDOM_SEED, **rf_best_params),
    X_german_train, y_german_train, skf, "RF (tuned, no resampling)"
)
result_xgb_final = cv_evaluate(
    XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_best_params),
    X_german_train, y_german_train, skf, "XGBoost (tuned, no resampling)"
)

# Tuned stacked ensemble, no resampling
stack_plain = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_best_params)),
        ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_best_params))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=skf
)
result_stack_final = cv_evaluate(stack_plain, X_german_train, y_german_train, skf, "Stacked (tuned, no resampling)")

print(pd.DataFrame([result_rf_final, result_xgb_final, result_stack_final]))

                           config          AUC-ROC               F1  \
0       RF (tuned, no resampling)  0.790 +/- 0.039  0.481 +/- 0.103   
1  XGBoost (tuned, no resampling)  0.785 +/- 0.041  0.579 +/- 0.066   
2  Stacked (tuned, no resampling)  0.796 +/- 0.039  0.500 +/- 0.082   

            G-mean              MCC  
0  0.589 +/- 0.083  0.357 +/- 0.114  
1  0.679 +/- 0.050  0.424 +/- 0.090  
2  0.605 +/- 0.067  0.372 +/- 0.091  


In [11]:
# ============================================================
# DATASET: German Credit
# Step: Tuned RF/XGBoost/Stack with SMOTE, full CV evaluation
# ============================================================

rf_smote_best = {k.replace("clf__", ""): v for k, v in rf_smote_grid.best_params_.items()}
xgb_smote_best = {k.replace("clf__", ""): v for k, v in xgb_smote_grid.best_params_.items()}

rf_smote_final = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smote_best))
])
result_rf_smote_final = cv_evaluate(rf_smote_final, X_german_train, y_german_train, skf, "RF (tuned, SMOTE)")

xgb_smote_final = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smote_best))
])
result_xgb_smote_final = cv_evaluate(xgb_smote_final, X_german_train, y_german_train, skf, "XGBoost (tuned, SMOTE)")

stack_smote_final = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smote_best)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smote_best))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smote_final = cv_evaluate(stack_smote_final, X_german_train, y_german_train, skf, "Stacked (tuned, SMOTE)")

print(pd.DataFrame([result_rf_smote_final, result_xgb_smote_final, result_stack_smote_final]))

                   config          AUC-ROC               F1           G-mean  \
0       RF (tuned, SMOTE)  0.791 +/- 0.042  0.591 +/- 0.091  0.694 +/- 0.073   
1  XGBoost (tuned, SMOTE)  0.772 +/- 0.037  0.607 +/- 0.050  0.720 +/- 0.043   
2  Stacked (tuned, SMOTE)  0.794 +/- 0.044  0.582 +/- 0.083  0.689 +/- 0.068   

               MCC  
0  0.430 +/- 0.116  
1  0.416 +/- 0.073  
2  0.410 +/- 0.106  


In [12]:
# ============================================================
# DATASET: German Credit
# Step: Tuned RF/XGBoost/Stack with SMOTE-ENN, full CV evaluation
# ============================================================

rf_smoteenn_best = {k.replace("clf__", ""): v for k, v in rf_smoteenn_grid.best_params_.items()}
xgb_smoteenn_best = {k.replace("clf__", ""): v for k, v in xgb_smoteenn_grid.best_params_.items()}

rf_smoteenn_final = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smoteenn_best))
])
result_rf_smoteenn_final = cv_evaluate(rf_smoteenn_final, X_german_train, y_german_train, skf, "RF (tuned, SMOTE-ENN)")

xgb_smoteenn_final = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smoteenn_best))
])
result_xgb_smoteenn_final = cv_evaluate(xgb_smoteenn_final, X_german_train, y_german_train, skf, "XGBoost (tuned, SMOTE-ENN)")

stack_smoteenn_final = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smoteenn_best)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smoteenn_best))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smoteenn_final = cv_evaluate(stack_smoteenn_final, X_german_train, y_german_train, skf, "Stacked (tuned, SMOTE-ENN)")

print(pd.DataFrame([result_rf_smoteenn_final, result_xgb_smoteenn_final, result_stack_smoteenn_final]))

                       config          AUC-ROC               F1  \
0       RF (tuned, SMOTE-ENN)  0.778 +/- 0.044  0.602 +/- 0.029   
1  XGBoost (tuned, SMOTE-ENN)  0.778 +/- 0.046  0.610 +/- 0.036   
2  Stacked (tuned, SMOTE-ENN)  0.786 +/- 0.044  0.600 +/- 0.037   

            G-mean              MCC  
0  0.713 +/- 0.026  0.397 +/- 0.050  
1  0.722 +/- 0.032  0.412 +/- 0.060  
2  0.714 +/- 0.032  0.397 +/- 0.060  


In [13]:
# ============================================================
# DATASET: LendingClub
# Purpose: Hyperparameter tuning + full 5-fold CV, same process as German Credit
# ============================================================

X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
clean_column_names(X_lending_train)
X_lending_test = pd.read_csv(SCALED_DIR / "lending_X_test.csv")
clean_column_names(X_lending_test)
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()
y_lending_test = pd.read_csv(SCALED_DIR / "lending_y_test.csv").squeeze()

print(X_lending_train.shape, X_lending_test.shape)
print("Minority cases in training:", (y_lending_train == 1).sum())

(8000, 69) (2000, 69)
Minority cases in training: 142


In [14]:
# ============================================================
# DATASET: LendingClub
# Step: Grid search RF - no resampling, SMOTE, SMOTE-ENN
# ============================================================

rf_grid_lc = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED),
    param_grid=rf_param_grid, scoring="f1", cv=skf, n_jobs=-1
)
rf_grid_lc.fit(X_lending_train, y_lending_train)
print("Best RF params (no resampling):", rf_grid_lc.best_params_, "| F1:", rf_grid_lc.best_score_)

rf_smote_pipe_lc = ImbPipeline([("smote", SMOTE(random_state=RANDOM_SEED)), ("clf", RandomForestClassifier(random_state=RANDOM_SEED))])
rf_smote_grid_lc = GridSearchCV(rf_smote_pipe_lc, param_grid=rf_smote_param_grid, scoring="f1", cv=skf, n_jobs=-1)
rf_smote_grid_lc.fit(X_lending_train, y_lending_train)
print("Best RF+SMOTE params:", rf_smote_grid_lc.best_params_, "| F1:", rf_smote_grid_lc.best_score_)

rf_smoteenn_pipe_lc = ImbPipeline([("smoteenn", SMOTEENN(random_state=RANDOM_SEED)), ("clf", RandomForestClassifier(random_state=RANDOM_SEED))])
rf_smoteenn_grid_lc = GridSearchCV(rf_smoteenn_pipe_lc, param_grid=rf_smote_param_grid, scoring="f1", cv=skf, n_jobs=-1)
rf_smoteenn_grid_lc.fit(X_lending_train, y_lending_train)
print("Best RF+SMOTE-ENN params:", rf_smoteenn_grid_lc.best_params_, "| F1:", rf_smoteenn_grid_lc.best_score_)

Best RF params (no resampling): {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100} | F1: 0.11830923248053393
Best RF+SMOTE params: {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 10, 'clf__n_estimators': 300} | F1: 0.13514371860696953
Best RF+SMOTE-ENN params: {'clf__max_depth': 20, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 10, 'clf__n_estimators': 100} | F1: 0.14725072120420962


In [15]:
# ============================================================
# DATASET: LendingClub
# Step: Grid search XGBoost - no resampling, SMOTE, SMOTE-ENN
# ============================================================

xgb_grid_lc = GridSearchCV(
    XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"),
    param_grid=xgb_param_grid, scoring="f1", cv=skf, n_jobs=-1
)
xgb_grid_lc.fit(X_lending_train, y_lending_train)
print("Best XGBoost params (no resampling):", xgb_grid_lc.best_params_, "| F1:", xgb_grid_lc.best_score_)

xgb_smote_pipe_lc = ImbPipeline([("smote", SMOTE(random_state=RANDOM_SEED)), ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))])
xgb_smote_grid_lc = GridSearchCV(xgb_smote_pipe_lc, param_grid=xgb_smote_param_grid, scoring="f1", cv=skf, n_jobs=-1)
xgb_smote_grid_lc.fit(X_lending_train, y_lending_train)
print("Best XGBoost+SMOTE params:", xgb_smote_grid_lc.best_params_, "| F1:", xgb_smote_grid_lc.best_score_)

xgb_smoteenn_pipe_lc = ImbPipeline([("smoteenn", SMOTEENN(random_state=RANDOM_SEED)), ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))])
xgb_smoteenn_grid_lc = GridSearchCV(xgb_smoteenn_pipe_lc, param_grid=xgb_smote_param_grid, scoring="f1", cv=skf, n_jobs=-1)
xgb_smoteenn_grid_lc.fit(X_lending_train, y_lending_train)
print("Best XGBoost+SMOTE-ENN params:", xgb_smoteenn_grid_lc.best_params_, "| F1:", xgb_smoteenn_grid_lc.best_score_)

Best XGBoost params (no resampling): {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8} | F1: 0.40317965023847374
Best XGBoost+SMOTE params: {'clf__colsample_bytree': 1.0, 'clf__learning_rate': 0.2, 'clf__max_depth': 3, 'clf__n_estimators': 300, 'clf__subsample': 0.8} | F1: 0.38855066502125324
Best XGBoost+SMOTE-ENN params: {'clf__colsample_bytree': 1.0, 'clf__learning_rate': 0.2, 'clf__max_depth': 3, 'clf__n_estimators': 300, 'clf__subsample': 0.8} | F1: 0.3780762059934075


In [16]:
# ============================================================
# DATASET: LendingClub
# Step: Full CV evaluation (all 4 metrics) using tuned models,
# no resampling condition
# ============================================================

rf_best_lc = rf_grid_lc.best_params_
xgb_best_lc = xgb_grid_lc.best_params_

result_rf_final_lc = cv_evaluate(
    RandomForestClassifier(random_state=RANDOM_SEED, **rf_best_lc),
    X_lending_train, y_lending_train, skf, "RF (tuned, no resampling)"
)
result_xgb_final_lc = cv_evaluate(
    XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_best_lc),
    X_lending_train, y_lending_train, skf, "XGBoost (tuned, no resampling)"
)

stack_plain_lc = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_best_lc)),
        ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_best_lc))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=skf
)
result_stack_final_lc = cv_evaluate(stack_plain_lc, X_lending_train, y_lending_train, skf, "Stacked (tuned, no resampling)")

print(pd.DataFrame([result_rf_final_lc, result_xgb_final_lc, result_stack_final_lc]))

                           config          AUC-ROC               F1  \
0       RF (tuned, no resampling)  0.772 +/- 0.047  0.118 +/- 0.047   
1  XGBoost (tuned, no resampling)  0.873 +/- 0.040  0.403 +/- 0.116   
2  Stacked (tuned, no resampling)  0.835 +/- 0.050  0.365 +/- 0.132   

            G-mean              MCC  
0  0.246 +/- 0.053  0.244 +/- 0.053  
1  0.504 +/- 0.092  0.486 +/- 0.100  
2  0.471 +/- 0.108  0.454 +/- 0.116  


In [17]:
rf_smote_best_lc = {k.replace("clf__", ""): v for k, v in rf_smote_grid_lc.best_params_.items()}
xgb_smote_best_lc = {k.replace("clf__", ""): v for k, v in xgb_smote_grid_lc.best_params_.items()}

rf_smote_final_lc = ImbPipeline([("smote", SMOTE(random_state=RANDOM_SEED)), ("clf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smote_best_lc))])
result_rf_smote_final_lc = cv_evaluate(rf_smote_final_lc, X_lending_train, y_lending_train, skf, "RF (tuned, SMOTE)")

xgb_smote_final_lc = ImbPipeline([("smote", SMOTE(random_state=RANDOM_SEED)), ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smote_best_lc))])
result_xgb_smote_final_lc = cv_evaluate(xgb_smote_final_lc, X_lending_train, y_lending_train, skf, "XGBoost (tuned, SMOTE)")

stack_smote_final_lc = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smote_best_lc)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smote_best_lc))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smote_final_lc = cv_evaluate(stack_smote_final_lc, X_lending_train, y_lending_train, skf, "Stacked (tuned, SMOTE)")

print(pd.DataFrame([result_rf_smote_final_lc, result_xgb_smote_final_lc, result_stack_smote_final_lc]))

                   config          AUC-ROC               F1           G-mean  \
0       RF (tuned, SMOTE)  0.769 +/- 0.010  0.135 +/- 0.096  0.352 +/- 0.148   
1  XGBoost (tuned, SMOTE)  0.858 +/- 0.036  0.389 +/- 0.148  0.497 +/- 0.124   
2  Stacked (tuned, SMOTE)  0.798 +/- 0.023  0.361 +/- 0.179  0.481 +/- 0.156   

               MCC  
0  0.121 +/- 0.099  
1  0.455 +/- 0.127  
2  0.407 +/- 0.167  


In [18]:
# ============================================================
# DATASET: LendingClub
# Step: Tuned RF/XGBoost/Stack with SMOTE-ENN, full CV evaluation
# ============================================================

rf_smoteenn_best_lc = {k.replace("clf__", ""): v for k, v in rf_smoteenn_grid_lc.best_params_.items()}
xgb_smoteenn_best_lc = {k.replace("clf__", ""): v for k, v in xgb_smoteenn_grid_lc.best_params_.items()}

rf_smoteenn_final_lc = ImbPipeline([("smoteenn", SMOTEENN(random_state=RANDOM_SEED)), ("clf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smoteenn_best_lc))])
result_rf_smoteenn_final_lc = cv_evaluate(rf_smoteenn_final_lc, X_lending_train, y_lending_train, skf, "RF (tuned, SMOTE-ENN)")

xgb_smoteenn_final_lc = ImbPipeline([("smoteenn", SMOTEENN(random_state=RANDOM_SEED)), ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smoteenn_best_lc))])
result_xgb_smoteenn_final_lc = cv_evaluate(xgb_smoteenn_final_lc, X_lending_train, y_lending_train, skf, "XGBoost (tuned, SMOTE-ENN)")

stack_smoteenn_final_lc = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_smoteenn_best_lc)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_smoteenn_best_lc))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smoteenn_final_lc = cv_evaluate(stack_smoteenn_final_lc, X_lending_train, y_lending_train, skf, "Stacked (tuned, SMOTE-ENN)")

print(pd.DataFrame([result_rf_smoteenn_final_lc, result_xgb_smoteenn_final_lc, result_stack_smoteenn_final_lc]))

                       config          AUC-ROC               F1  \
0       RF (tuned, SMOTE-ENN)  0.785 +/- 0.014  0.147 +/- 0.104   
1  XGBoost (tuned, SMOTE-ENN)  0.850 +/- 0.028  0.378 +/- 0.140   
2  Stacked (tuned, SMOTE-ENN)  0.808 +/- 0.021  0.329 +/- 0.134   

            G-mean              MCC  
0  0.302 +/- 0.120  0.152 +/- 0.108  
1  0.491 +/- 0.119  0.447 +/- 0.114  
2  0.460 +/- 0.122  0.376 +/- 0.122  


In [19]:
# ============================================================
# DATASET: Home Credit
# Purpose: Hyperparameter tuning + full 5-fold CV
# Note: using RandomizedSearchCV (20 iterations) instead of full
# GridSearchCV here specifically, given this dataset's size
# (246,005 training rows) and the memory constraints already
# encountered on this dataset during SHAP analysis (Step 6)
# ============================================================

from sklearn.model_selection import RandomizedSearchCV

X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
clean_column_names(X_home_train)
X_home_test = pd.read_csv(SCALED_DIR / "home_X_test.csv")
clean_column_names(X_home_test)
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()
y_home_test = pd.read_csv(SCALED_DIR / "home_y_test.csv").squeeze()

print(X_home_train.shape, X_home_test.shape)

(246005, 100) (61502, 100)


In [20]:
# ============================================================
# DATASET: Home Credit
# Step: Randomized search RF, no resampling (20 iterations instead
# of exhaustive grid, to manage compute time on this dataset's scale)
# ============================================================

rf_random_home = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED),
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="f1",
    cv=skf,
    n_jobs=-1,
    random_state=RANDOM_SEED
)
rf_random_home.fit(X_home_train, y_home_train)

print("Best RF params (no resampling):", rf_random_home.best_params_)
print("Best CV F1 score:", rf_random_home.best_score_)

Best RF params (no resampling): {'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}
Best CV F1 score: 0.0009054832519406886


In [3]:
scoring = {
    "AUC-ROC": "roc_auc",
    "F1": "f1",
    "G-mean": make_scorer(geometric_mean_score),
    "MCC": make_scorer(matthews_corrcoef)
}

rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2"]
}

xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 6, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [ ]:
X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
clean_column_names(X_home_train)
X_home_test = pd.read_csv(SCALED_DIR / "home_X_test.csv")
clean_column_names(X_home_test)
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()
y_home_test = pd.read_csv(SCALED_DIR / "home_y_test.csv").squeeze()

print(X_home_train.shape, X_home_test.shape)

(246005, 100) (61502, 100)


In [6]:
# ============================================================
# DATASET: Home Credit — full rebuild after kernel restart
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import make_scorer, roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier

SCALED_DIR = Path("../Data/Processed")
RANDOM_SEED = 42

def clean_column_names(df):
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
skf_light = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    "AUC-ROC": "roc_auc", "F1": "f1",
    "G-mean": make_scorer(geometric_mean_score),
    "MCC": make_scorer(matthews_corrcoef)
}

rf_param_grid = {
    "n_estimators": [100, 200, 300], "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10], "max_features": ["sqrt", "log2"]
}
xgb_param_grid = {
    "n_estimators": [100, 200, 300], "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 6, 10], "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]
}

# Reload Home Credit
X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
clean_column_names(X_home_train)
X_home_test = pd.read_csv(SCALED_DIR / "home_X_test.csv")
clean_column_names(X_home_test)
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()
y_home_test = pd.read_csv(SCALED_DIR / "home_y_test.csv").squeeze()
print("Data reloaded:", X_home_train.shape, X_home_test.shape)

# Re-run RF no-resample and RF+SMOTE (both completed quickly before)
rf_random_home = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED),
    param_distributions=rf_param_grid, n_iter=20, scoring="f1",
    cv=skf, n_jobs=-1, random_state=RANDOM_SEED
)
rf_random_home.fit(X_home_train, y_home_train)
print("RF (no resample) done:", rf_random_home.best_params_, rf_random_home.best_score_)

rf_smote_pipe_home = ImbPipeline([("smote", SMOTE(random_state=RANDOM_SEED)), ("clf", RandomForestClassifier(random_state=RANDOM_SEED))])
rf_smote_param_grid_home = {f"clf__{k}": v for k, v in rf_param_grid.items()}
rf_smote_random_home = RandomizedSearchCV(
    rf_smote_pipe_home, param_distributions=rf_smote_param_grid_home,
    n_iter=20, scoring="f1", cv=skf, n_jobs=-1, random_state=RANDOM_SEED
)
rf_smote_random_home.fit(X_home_train, y_home_train)
print("RF+SMOTE done:", rf_smote_random_home.best_params_, rf_smote_random_home.best_score_)

# RF+SMOTE-ENN - scoped down (5 iterations, 3-fold) given the earlier 14+ hour runtime
rf_smoteenn_pipe_home = ImbPipeline([("smoteenn", SMOTEENN(random_state=RANDOM_SEED)), ("clf", RandomForestClassifier(random_state=RANDOM_SEED))])
rf_smoteenn_random_home = RandomizedSearchCV(
    rf_smoteenn_pipe_home, param_distributions=rf_smote_param_grid_home,
    n_iter=5, scoring="f1", cv=skf_light, n_jobs=-1, random_state=RANDOM_SEED
)
rf_smoteenn_random_home.fit(X_home_train, y_home_train)
print("RF+SMOTE-ENN done:", rf_smoteenn_random_home.best_params_, rf_smoteenn_random_home.best_score_)

Data reloaded: (246005, 100) (61502, 100)
RF (no resample) done: {'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20} 0.0009054832519406886
RF+SMOTE done: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 10, 'clf__max_features': 'log2', 'clf__max_depth': 10} 0.20531434947035104


c:\Users\Thokozani\Desktop\BcomHons\Second Semester\Information Systems Research\Project\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


RF+SMOTE-ENN done: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 5, 'clf__max_features': 'log2', 'clf__max_depth': 20} 0.24454019255694692


In [7]:
# ============================================================
# DATASET: Home Credit
# Step: Randomized search XGBoost - all three conditions,
# using light settings (5 iter, 3-fold) throughout given the
# confirmed heavy runtime cost on this dataset
# ============================================================

xgb_random_home = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"),
    param_distributions=xgb_param_grid, n_iter=5, scoring="f1",
    cv=skf_light, n_jobs=-1, random_state=RANDOM_SEED
)
xgb_random_home.fit(X_home_train, y_home_train)
print("XGBoost (no resample) done:", xgb_random_home.best_params_, xgb_random_home.best_score_)

xgb_smote_pipe_home = ImbPipeline([("smote", SMOTE(random_state=RANDOM_SEED)), ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))])
xgb_smote_param_grid_home = {f"clf__{k}": v for k, v in xgb_param_grid.items()}
xgb_smote_random_home = RandomizedSearchCV(
    xgb_smote_pipe_home, param_distributions=xgb_smote_param_grid_home,
    n_iter=5, scoring="f1", cv=skf_light, n_jobs=-1, random_state=RANDOM_SEED
)
xgb_smote_random_home.fit(X_home_train, y_home_train)
print("XGBoost+SMOTE done:", xgb_smote_random_home.best_params_, xgb_smote_random_home.best_score_)

xgb_smoteenn_pipe_home = ImbPipeline([("smoteenn", SMOTEENN(random_state=RANDOM_SEED)), ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))])
xgb_smoteenn_random_home = RandomizedSearchCV(
    xgb_smoteenn_pipe_home, param_distributions=xgb_smote_param_grid_home,
    n_iter=5, scoring="f1", cv=skf_light, n_jobs=-1, random_state=RANDOM_SEED
)
xgb_smoteenn_random_home.fit(X_home_train, y_home_train)
print("XGBoost+SMOTE-ENN done:", xgb_smoteenn_random_home.best_params_, xgb_smoteenn_random_home.best_score_)

XGBoost (no resample) done: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0} 0.04075577720450784
XGBoost+SMOTE done: {'clf__subsample': 0.8, 'clf__n_estimators': 300, 'clf__max_depth': 3, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.8} 0.189731025162961
XGBoost+SMOTE-ENN done: {'clf__subsample': 0.8, 'clf__n_estimators': 300, 'clf__max_depth': 6, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.8} 0.24197052794533977


In [ ]:
# ============================================================
# DATASET: Home Credit
# Step: Final tuned CV evaluation (all 4 metrics), 3-fold given
# this dataset's confirmed compute cost - applied consistently
# across all 9 configurations for comparability
# ============================================================

def cv_evaluate_light(model, X, y, cv, name):
    results = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return {
        "config": name,
        "AUC-ROC": f"{results['test_AUC-ROC'].mean():.3f} +/- {results['test_AUC-ROC'].std():.3f}",
        "F1": f"{results['test_F1'].mean():.3f} +/- {results['test_F1'].std():.3f}",
        "G-mean": f"{results['test_G-mean'].mean():.3f} +/- {results['test_G-mean'].std():.3f}",
        "MCC": f"{results['test_MCC'].mean():.3f} +/- {results['test_MCC'].std():.3f}"
    }

# Extract best params
rf_best_home = rf_random_home.best_params_
rf_smote_best_home = {k.replace("clf__", ""): v for k, v in rf_smote_random_home.best_params_.items()}
rf_smoteenn_best_home = {k.replace("clf__", ""): v for k, v in rf_smoteenn_random_home.best_params_.items()}
xgb_best_home = xgb_random_home.best_params_
xgb_smote_best_home = {k.replace("clf__", ""): v for k, v in xgb_smote_random_home.best_params_.items()}
xgb_smoteenn_best_home = {k.replace("clf__", ""): v for k, v in xgb_smoteenn_random_home.best_params_.items()}

home_final_results = []

# No resampling
home_final_results.append(cv_evaluate_light(RandomForestClassifier(random_state=RANDOM_SEED, **rf_best_home), X_home_train, y_home_train, skf_light, "RF (tuned, no resampling)"))
home_final_results.append(cv_evaluate_light(XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_best_home), X_home_train, y_home_train, skf_light, "XGBoost (tuned, no resampling)"))

stack_plain_home = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=RANDOM_SEED, **rf_best_home)),
        ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", **xgb_best_home))
    ],
    final_estimator=LogisticRegression(max_iter=1000), cv=skf_light
)
home_final_results.append(cv_evaluate_light(stack_plain_home, X_home_train, y_home_train, skf_light, "Stacked (tuned, no resampling)"))

print(pd.DataFrame(home_final_results))